You’re given access to a few custom functions(helper_funcitons.py) and you’re expected to use the relevant one from the Git repository. Please ensure you only use the provided seed values and don’t modify those cells as changing them will alter the result.

### Please use the competition dataset for this assignment.

* https://github.com/Photon-08/milestone-4-codebase/tree/main

# - Code

In [1]:
import torch
import numpy as np
import random
import torchaudio
import os
import glob
from pathlib import Path

# --- SET YOUR KAGGLE PATHS ---
INPUT_BASE = '/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup'
WORKING_BASE = '/kaggle/working'

STEMS_PATH = os.path.join(INPUT_BASE, 'genres_stems')
NOISE_PATH = os.path.join(INPUT_BASE, 'ESC-50-master/audio')
OUTPUT_PATH = os.path.join(WORKING_BASE, 'synthetic_mashups/train')


def seed_everything(seed=42):
    """Locks all random seeds for absolute reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    
    # If using GPU
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        # Forces deterministic algorithms
        torch.backends.cudnn.deterministic = True 
        torch.backends.cudnn.benchmark = False

# Execute immediately at the top of the script
seed_everything(42)







def generate_synthetic_dataset(stems_dir, noise_dir, output_dir, samples_per_genre=50, target_sr=22050, duration=30):
    """Generates deterministic noisy mashups and saves them to /kaggle/working/."""
    genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"
]
    target_length = target_sr * duration
    
    # Get noise files from read-only input
    noise_files = glob.glob(os.path.join(noise_dir, '**', '*.wav'), recursive=True)
    
    for genre in genres:
        # Create output directories in the writable /kaggle/working/ directory
        genre_out_dir = Path(output_dir) / genre
        genre_out_dir.mkdir(parents=True, exist_ok=True)
        
        song_folders = glob.glob(os.path.join(stems_dir, genre, '*'))
        if not song_folders: 
            print(f"Warning: No songs found for genre {genre}")
            continue
        
        for i in range(samples_per_genre):
            chosen_songs = random.sample(song_folders, 4)
            stems = []
            stem_types = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']
            
            for song, stem_type in zip(chosen_songs, stem_types):
                stem_path = os.path.join(song, stem_type)
                if os.path.exists(stem_path):
                    waveform, sr = torchaudio.load(stem_path)
                    
                    # Basic Resampling check (if needed)
                    if sr != target_sr:
                        resampler = torchaudio.transforms.Resample(sr, target_sr)
                        waveform = resampler(waveform)

                    if waveform.shape[1] > target_length:
                        waveform = waveform[:, :target_length]
                    elif waveform.shape[1] < target_length:
                        waveform = torch.nn.functional.pad(waveform, (0, target_length - waveform.shape[1]))
                    stems.append(waveform)
            
            if len(stems) == 4:
                mashup = torch.stack(stems).sum(dim=0)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                noise_file = random.choice(noise_files)
                noise, _ = torchaudio.load(noise_file)
                
                if noise.shape[1] > target_length:
                    noise = noise[:, :target_length]
                    
                start_idx = random.randint(0, target_length - noise.shape[1])
                intensity = random.uniform(0.1, 0.4)
                
                mashup[:, start_idx:start_idx + noise.shape[1]] += (noise * intensity)
                mashup = mashup / (torch.max(torch.abs(mashup)) + 1e-8)
                
                # Save to /kaggle/working/
                out_path = genre_out_dir / f"mashup_{i:03d}.wav"
                torchaudio.save(str(out_path), mashup, target_sr)

# Run the generation
generate_synthetic_dataset(STEMS_PATH, NOISE_PATH, OUTPUT_PATH, samples_per_genre=50)




import os
import glob
import torch
import torchaudio
from pathlib import Path

def extract_and_save_features(input_dir, output_dir, target_sr=22050):
    """Converts audio to Mel-spectrograms in dB and saves as PyTorch tensors."""
    mel_transform = torchaudio.transforms.MelSpectrogram(
        sample_rate=target_sr, n_fft=2048, hop_length=512, n_mels=128
    )
    amplitude_to_db = torchaudio.transforms.AmplitudeToDB()

    # Find all .wav files in the input directory
    wav_files = glob.glob(os.path.join(input_dir, '**', '*.wav'), recursive=True)
    
    if not wav_files:
        print(f"Warning: No .wav files found in {input_dir}")
        return

    for wav_path in wav_files:
        # Replicate directory structure
        rel_path = os.path.relpath(wav_path, input_dir)
        out_path = Path(output_dir) / rel_path
        out_path = out_path.with_suffix('.pt')
        
        # Ensure the target directory exists in /kaggle/working/
        out_path.parent.mkdir(parents=True, exist_ok=True)
        
        # Process and save
        waveform, sr = torchaudio.load(wav_path)
        mel_spec = mel_transform(waveform)
        mel_spec_db = amplitude_to_db(mel_spec)
        
        torch.save(mel_spec_db, out_path)
    
    print(f"Successfully saved {len(wav_files)} feature files to {output_dir}")


INPUT_DIR = '/kaggle/working/synthetic_mashups/train'
OUTPUT_DIR = '/kaggle/working/features/train'

extract_and_save_features(INPUT_DIR, OUTPUT_DIR)

Successfully saved 500 feature files to /kaggle/working/features/train


In [2]:
# Q1) You ran the generate_synthetic_dataset script with samples_per_genre=50. 
# If the script executed successfully across all 10 target genres, 
# exactly how many .wav files should now exist in your /kaggle/working/synthetic_mashups/train/ directory tree?
count = 0
genres = ["blues", "classical", "country", "disco", "hiphop",
"jazz", "metal", "pop", "reggae", "rock"]
for g in genres:
    for dirname, _, filenames in os.walk(f'/kaggle/working/synthetic_mashups/train/{g}'):
        for filename in filenames:
            count += 1
print(count)

500


In [3]:
# Q2) Load any of your newly generated .wav files using torchaudio.load(). 
# Given our configuration (target_sr=22050, duration=30), what is the exact tensor shape of the resulting waveform?
# Provide in tuple format, example (x,y)
f = f'/kaggle/working/synthetic_mashups/train/pop/mashup_001.wav'
y,sr = torchaudio.load("/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup/genres_stems/blues/blues.00000/bass.wav")
y.size()

torch.Size([2, 1323588])

In [4]:
# Q3) Using the extract_and_save_features script with n_fft=2048, hop_length=512, and n_mels=128, you converted the waveforms into .pt files. 
#If you load one of these pre-computed tensors using torch.load(), what is its exact dimension shape? 

mel_spec = torch.load("/kaggle/working/features/train/blues/mashup_000.pt")
mono_mel_spec = torch.mean(mel_spec,dim=0).unsqueeze(0)
mel_spec.size(),mono_mel_spec.size()

(torch.Size([2, 128, 1292]), torch.Size([1, 128, 1292]))

### Build Your Model (The CRNN)

* Goal: You will build a model that understands both what sounds are playing (using a CNN) and the rhythm of how they play over  time (using an RNN).

### What to Code:

* Create a PyTorch class called CRNN with these exact pieces:

    1. The Input Your model will take in a 1-channel Mel-spectrogram (treat it like a single-color image).

    2. The CNN (Sound Feature Extractor)

        * Write two blocks of layers to find patterns in the audio frequencies:
        
            * Block 1: Conv2D (32 filters, $3 \times 3$ kernel, padding 1) $\rightarrow$ BatchNorm2d $\rightarrow$ ReLU $\rightarrow$           MaxPool 2d  ($2 \times 2$ window).
        
            * Block 2: Conv2D (64 filters, $3 \times 3$ kernel, padding 1) ---> BatchNorm2d ---> ReLU ---> MaxPool2d ($2 \times 2$ window).

    3. The Bridge (Reshaping)

        * The CNN outputs a 3D block of data (channels, frequencies, and time). To pass this into an RNN, you must reshape it. Keep the time steps as your sequence, and flatten the channels and frequencies together into one single feature list per time step.

    4. The RNN (Rhythm Tracker)

        * Pass your reshaped sequence into a 1-layer Bidirectional LSTM. (Set hidden_size=64 and batch_first=True).

    5. The Final Prediction
        * The LSTM outputs data for every single time step. To get one final answer for the whole 30-second song, take the maximum value across all time steps (known as Global Max Pooling). Finally, pass that summary vector through a standard Linear layer to output the 10 genre scores.

In [45]:
# Model
import torch.nn as nn
class CRNN(nn.Module):
    def __init__(self,num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            # Block:1
            nn.Conv2d(
                in_channels =1,
                out_channels = 32,
                kernel_size = 3,
                padding = 1
            ),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((2,2)),

            #Block:2
            nn.Conv2d(
                in_channels=32,
                out_channels = 64,
                kernel_size = 3,
                padding = 1
            ),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d((2,2))
        )

        self.lstm = nn.LSTM(
            2048,
            hidden_size=64,
            batch_first=True,
            num_layers=1,
            bidirectional=True
        )
        self.fc = nn.Linear(64*2, num_classes)
    def forward(self,x):
        #x : (B,1,128,1292)
        x_cnn = self.cnn(x) 
        #x_cnn : (B,64,32,323)
        batch,channel,freq,time = x_cnn.size()
        x_rnn_in = x_cnn.permute(0, 3, 1, 2).reshape(batch,time,channel*freq)
        #x_rnn_in : (B,323,2048)
        x_rnn_out, h_s, c_s = self.rnn(x_rnn_in)
        #x_rnn_out : (B,323,128)     128 = 2*64[bidir = 2, hid_size=64]

        pooled_x,_ = torch.max(x_rnn_out, dim=1)
        #pooled_x : (B,128)
        out_logit = self.fc(pooled_x)        
        #out_logit : (B,10)
        return out_logit

In [46]:
# 4) Inside the CRNN model's forward pass, the data moves from the CNN backbone to the LSTM. 
# If your input batch has a size of 32, 
# what is the exact shape of the tensor immediately after the second MaxPool2d(2) layer, right before the .permute() operation?
# SHAPE : [32,64,32,323]

model = CRNN(10)

In [52]:
# Q5) Your model uses a Bidirectional LSTM with input_size=2048 and hidden_size=64. 
# Write a small snippet using sum(p.numel() for p in model.lstm.parameters() if p.requires_grad) 
# to calculate the exact number of trainable parameters in the LSTM layer alone. 
# What is that integer value?

#sum(p.numel() for p in model.lstm.parameters() if p.requires_grad)
for name,p in model.lstm.named_parameters() :
    if p.requires_grad:
        print(name," - ",p.size())

weight_ih_l0  -  torch.Size([256, 2048])
weight_hh_l0  -  torch.Size([256, 64])
bias_ih_l0  -  torch.Size([256])
bias_hh_l0  -  torch.Size([256])
weight_ih_l0_reverse  -  torch.Size([256, 2048])
weight_hh_l0_reverse  -  torch.Size([256, 64])
bias_ih_l0_reverse  -  torch.Size([256])
bias_hh_l0_reverse  -  torch.Size([256])


# - Now train the model using the following configuraiton:

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CRNN(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)



train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

num_epochs = 10

In [ ]:
# Q6) When applying a 2D CNN to a Mel-spectrogram, 
# translation invariance along the X-axis (Time) is highly beneficial, 
# as a guitar solo at 0:15 is the same as a guitar solo at 0:25. 
# However, why is translation invariance along the Y-axis (Mel-Frequency Bins) conceptually problematic for music?